In [6]:
# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas

In [7]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.444,
	"longitude": 3.703,
	"start_date": "2020-01-10",
	"end_date": "2026-03-24",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "sunrise", "sunset", "daylight_duration", "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours", "surface_pressure_mean", "et0_fao_evapotranspiration_sum"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()
daily_apparent_temperature_mean = daily.Variables(4).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(6).ValuesAsNumpy()
daily_shortwave_radiation_sum = daily.Variables(7).ValuesAsNumpy()
daily_sunrise = daily.Variables(8).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(9).ValuesInt64AsNumpy()
daily_daylight_duration = daily.Variables(10).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(11).ValuesAsNumpy()
daily_rain_sum = daily.Variables(12).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(13).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(14).ValuesAsNumpy()
daily_surface_pressure_mean = daily.Variables(15).ValuesAsNumpy()
daily_et0_fao_evapotranspiration_sum = daily.Variables(16).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["rain_sum"] = daily_rain_sum
daily_data["snowfall_sum"] = daily_snowfall_sum
daily_data["precipitation_hours"] = daily_precipitation_hours
daily_data["surface_pressure_mean"] = daily_surface_pressure_mean
daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 40.456939697265625°N 3.681241273880005°E
Elevation: 0.0 m asl
Timezone difference to GMT+0: 0s

Daily data
                           date  weather_code  temperature_2m_mean  \
0    2020-01-10 00:00:00+00:00          51.0            13.681251   
1    2020-01-11 00:00:00+00:00          51.0            12.699998   
2    2020-01-12 00:00:00+00:00           1.0            12.650001   
3    2020-01-13 00:00:00+00:00           3.0            12.889584   
4    2020-01-14 00:00:00+00:00           2.0            14.039582   
...                        ...           ...                  ...   
2261 2026-03-20 00:00:00+00:00           1.0            14.095834   
2262 2026-03-21 00:00:00+00:00           3.0            13.479167   
2263 2026-03-22 00:00:00+00:00           2.0            13.527081   
2264 2026-03-23 00:00:00+00:00          61.0            13.258334   
2265 2026-03-24 00:00:00+00:00           0.0            14.085416   

      temperature_2m_max  temperature_2m_min  appa

In [8]:
extract_date = pd.to_datetime("today")
place = "madrid"
dates_from_to = "2020-2026"
csv_title = f"daily_{place}_{dates_from_to}_{extract_date.strftime('%Y-%m-%d')}_extracted.csv"

daily_dataframe.to_csv(csv_title, index = False)
daily_dataframe

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum
0,2020-01-10 00:00:00+00:00,51.0,13.681251,14.55,12.90,8.795880,51.781212,70.199997,5.16,1578640087,1578674224,34138.175781,1.1,1.1,0.0,7.0,1025.104004,1.898017
1,2020-01-11 00:00:00+00:00,51.0,12.699998,13.25,12.20,6.429699,50.412853,69.120003,7.91,1578726474,1578760686,34212.054688,0.1,0.1,0.0,1.0,1028.504028,2.351611
2,2020-01-12 00:00:00+00:00,1.0,12.650001,13.05,12.35,8.939696,27.595825,36.719997,9.54,1578812859,1578847147,34288.812500,0.0,0.0,0.0,0.0,1028.762573,1.923817
3,2020-01-13 00:00:00+00:00,3.0,12.889584,13.50,12.45,9.399581,29.627365,39.239998,8.79,1578899241,1578933609,34368.359375,0.0,0.0,0.0,0.0,1025.258423,1.880872
4,2020-01-14 00:00:00+00:00,2.0,14.039582,14.80,13.55,9.608594,32.714474,43.919998,8.93,1578985621,1579020071,34450.773438,0.0,0.0,0.0,0.0,1022.895813,2.348456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2261,2026-03-20 00:00:00+00:00,1.0,14.095834,14.65,13.75,10.313819,30.916540,39.239998,20.10,1773985705,1774029415,43711.652344,0.0,0.0,0.0,0.0,1015.325012,3.659553
2262,2026-03-21 00:00:00+00:00,3.0,13.479167,14.00,13.05,11.166500,19.665359,25.919998,19.76,1774072008,1774115878,43872.234375,0.0,0.0,0.0,0.0,1013.008484,3.042713
2263,2026-03-22 00:00:00+00:00,2.0,13.527081,14.20,13.00,11.649837,19.486610,26.280001,20.24,1774158311,1774202341,44032.027344,0.0,0.0,0.0,0.0,1011.850037,2.918771
2264,2026-03-23 00:00:00+00:00,61.0,13.258334,14.35,12.50,8.702656,47.376942,62.279995,16.27,1774244615,1774288804,44191.136719,3.5,3.5,0.0,9.0,1016.108215,2.434976


In [ ]:
#CODE:

# https://open-meteo.com/en/docs/historical-weather-api?start_date=2020-01-01&hourly=&daily=weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum&latitude=40.416,41.388,39.470,37.382,41.650&longitude=-3.703,2.158,-0.376,-5.973,-0.883&location_mode=csv_coordinates

In [11]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
city_names = ["barcelona", "valencia", "sevilla", "zaragoza"]
params = {
	"latitude": [41.388, 39.470, 37.382, 41.650],
	"longitude": [2.158, -0.376, -5.973, -0.883],
	"start_date": "2020-01-01",
	"end_date": "2026-03-24",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "sunrise", "sunset", "daylight_duration", "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours", "surface_pressure_mean", "et0_fao_evapotranspiration_sum"],
}
responses = openmeteo.weather_api(url, params = params)

city_frames = []

# Process all locations and collect a dataframe per city
for city_name, response in zip(city_names, responses):
	print(f"\nCity: {city_name}")
	print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_weather_code = daily.Variables(0).ValuesAsNumpy()
	daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
	daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
	daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()
	daily_apparent_temperature_mean = daily.Variables(4).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
	daily_wind_gusts_10m_max = daily.Variables(6).ValuesAsNumpy()
	daily_shortwave_radiation_sum = daily.Variables(7).ValuesAsNumpy()
	daily_sunrise = daily.Variables(8).ValuesInt64AsNumpy()
	daily_sunset = daily.Variables(9).ValuesInt64AsNumpy()
	daily_daylight_duration = daily.Variables(10).ValuesAsNumpy()
	daily_precipitation_sum = daily.Variables(11).ValuesAsNumpy()
	daily_rain_sum = daily.Variables(12).ValuesAsNumpy()
	daily_snowfall_sum = daily.Variables(13).ValuesAsNumpy()
	daily_precipitation_hours = daily.Variables(14).ValuesAsNumpy()
	daily_surface_pressure_mean = daily.Variables(15).ValuesAsNumpy()
	daily_et0_fao_evapotranspiration_sum = daily.Variables(16).ValuesAsNumpy()
	
	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}
	
	daily_data["weather_code"] = daily_weather_code
	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["temperature_2m_min"] = daily_temperature_2m_min
	daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
	daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
	daily_data["sunrise"] = daily_sunrise
	daily_data["sunset"] = daily_sunset
	daily_data["daylight_duration"] = daily_daylight_duration
	daily_data["precipitation_sum"] = daily_precipitation_sum
	daily_data["rain_sum"] = daily_rain_sum
	daily_data["snowfall_sum"] = daily_snowfall_sum
	daily_data["precipitation_hours"] = daily_precipitation_hours
	daily_data["surface_pressure_mean"] = daily_surface_pressure_mean
	daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum
	
	daily_city_df = pd.DataFrame(data = daily_data)
	daily_city_df["city"] = city_name
	city_frames.append(daily_city_df)

daily_dataframe_2 = pd.concat(city_frames, ignore_index = True)
daily_dataframe_2


City: barcelona
Coordinates: 41.441123962402344°N 2.2014386653900146°E
Elevation: 41.0 m asl
Timezone difference to GMT+0: 0s

City: valencia
Coordinates: 39.47275924682617°N -0.373443603515625°E
Elevation: 23.0 m asl
Timezone difference to GMT+0: 0s

City: sevilla
Coordinates: 37.36379623413086°N -5.97607421875°E
Elevation: 11.0 m asl
Timezone difference to GMT+0: 0s

City: zaragoza
Coordinates: 41.65201950073242°N -0.910430908203125°E
Elevation: 224.0 m asl
Timezone difference to GMT+0: 0s


,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum,city
0,2020-01-01 00:00:00+00:00,3.0,7.440250,11.921500,3.0715,5.172289,10.041354,16.199999,7.910000,1577863039,1577896291,33252.089844,0.0,0.0,0.0,0.0,1026.585205,0.978240,barcelona
1,2020-01-02 00:00:00+00:00,3.0,7.067333,12.871500,2.9715,4.512383,13.830749,26.280001,8.210000,1577949445,1577982743,33297.687500,0.0,0.0,0.0,0.0,1026.304688,1.009614,barcelona
2,2020-01-03 00:00:00+00:00,3.0,7.492333,12.721499,3.8715,4.790669,11.525623,21.240000,7.080000,1578035849,1578069196,33347.132812,0.0,0.0,0.0,0.0,1025.359253,0.989903,barcelona
3,2020-01-04 00:00:00+00:00,0.0,7.977749,14.271500,3.8715,4.953839,16.873980,28.799999,8.400000,1578122251,1578155651,33400.343750,0.0,0.0,0.0,0.0,1025.960815,1.343847,barcelona
4,2020-01-05 00:00:00+00:00,0.0,8.446502,15.071500,4.4215,5.358721,18.678415,32.760002,8.780000,1578208651,1578242108,33457.210938,0.0,0.0,0.0,0.0,1023.290710,1.439958,barcelona
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9095,2026-03-20 00:00:00+00:00,3.0,11.091666,17.049999,5.6000,7.920519,18.514643,42.480000,19.530001,1773986801,1774030522,43720.687500,0.0,0.0,0.0,0.0,988.567688,3.496094,zaragoza
9096,2026-03-21 00:00:00+00:00,3.0,10.772918,17.299999,4.5500,8.886056,7.421590,21.959999,20.059999,1774073100,1774116989,43888.179688,0.0,0.0,0.0,0.0,985.843079,2.908944,zaragoza
9097,2026-03-22 00:00:00+00:00,1.0,11.870833,18.950001,4.4000,8.145182,24.350615,52.919998,20.469999,1774159400,1774203455,44054.851562,0.0,0.0,0.0,0.0,984.697937,4.103381,zaragoza
9098,2026-03-23 00:00:00+00:00,3.0,12.535419,18.000000,8.0000,9.116881,19.669479,43.199997,16.750000,1774245700,1774289922,44220.800781,0.0,0.0,0.0,0.0,990.297424,3.307928,zaragoza


In [13]:
extract_date = pd.to_datetime("today")
place = "top5_cities"
dates_from_to = "2020-2026"
csv_title = f"daily_{place}_{dates_from_to}_{extract_date.strftime('%Y-%m-%d')}_extracted.csv"

daily_dataframe_2.to_csv(csv_title, index = False)
daily_dataframe_2

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum,city
0,2020-01-01 00:00:00+00:00,3.0,7.440250,11.921500,3.0715,5.172289,10.041354,16.199999,7.910000,1577863039,1577896291,33252.089844,0.0,0.0,0.0,0.0,1026.585205,0.978240,barcelona
1,2020-01-02 00:00:00+00:00,3.0,7.067333,12.871500,2.9715,4.512383,13.830749,26.280001,8.210000,1577949445,1577982743,33297.687500,0.0,0.0,0.0,0.0,1026.304688,1.009614,barcelona
2,2020-01-03 00:00:00+00:00,3.0,7.492333,12.721499,3.8715,4.790669,11.525623,21.240000,7.080000,1578035849,1578069196,33347.132812,0.0,0.0,0.0,0.0,1025.359253,0.989903,barcelona
3,2020-01-04 00:00:00+00:00,0.0,7.977749,14.271500,3.8715,4.953839,16.873980,28.799999,8.400000,1578122251,1578155651,33400.343750,0.0,0.0,0.0,0.0,1025.960815,1.343847,barcelona
4,2020-01-05 00:00:00+00:00,0.0,8.446502,15.071500,4.4215,5.358721,18.678415,32.760002,8.780000,1578208651,1578242108,33457.210938,0.0,0.0,0.0,0.0,1023.290710,1.439958,barcelona
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9095,2026-03-20 00:00:00+00:00,3.0,11.091666,17.049999,5.6000,7.920519,18.514643,42.480000,19.530001,1773986801,1774030522,43720.687500,0.0,0.0,0.0,0.0,988.567688,3.496094,zaragoza
9096,2026-03-21 00:00:00+00:00,3.0,10.772918,17.299999,4.5500,8.886056,7.421590,21.959999,20.059999,1774073100,1774116989,43888.179688,0.0,0.0,0.0,0.0,985.843079,2.908944,zaragoza
9097,2026-03-22 00:00:00+00:00,1.0,11.870833,18.950001,4.4000,8.145182,24.350615,52.919998,20.469999,1774159400,1774203455,44054.851562,0.0,0.0,0.0,0.0,984.697937,4.103381,zaragoza
9098,2026-03-23 00:00:00+00:00,3.0,12.535419,18.000000,8.0000,9.116881,19.669479,43.199997,16.750000,1774245700,1774289922,44220.800781,0.0,0.0,0.0,0.0,990.297424,3.307928,zaragoza
